# Embed, Cluster & Visualize Crops with BioClip 2

Pipeline:
1. **Background removal** with BiRefNet (rembg) — clean foreground, no reef/water context
2. **CLAHE** preprocessing — local contrast boost on the foreground
3. **BioClip 2 embeddings** — per-crop visual features
4. **Track-level averaging** — one embedding per track ID
5. **HDBSCAN clustering** + **UMAP** visualization

Expected input structure:
```
pipeline_output/
    <video_name>/
        id3_fish_f0042_best.jpg
        id3_fish_f0042_early.jpg
        ...
```


## 1. Install dependencies

In [1]:
# Core deps
%pip install -q open_clip_torch torch torchvision umap-learn hdbscan scikit-learn seaborn tqdm opencv-python

# rembg for background removal — GPU build falls back to CPU cleanly if no CUDA onnxruntime
%pip install -q "rembg[gpu]" 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 42.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 89.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 7.9 MB/s eta 0:00:00:00:010:04m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 133.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 102.5 MB/s eta 0:00:0000:01:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 97.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.1 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following depen

## 2. Imports

In [2]:
import numpy as np
import torch
import open_clip
import cv2
from PIL import Image
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

import hdbscan
import umap
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import silhouette_score

from rembg import remove, new_session

ImportError: cannot import name '_Ink' from 'PIL._typing' (/usr/local/lib/python3.12/dist-packages/PIL/_typing.py)

## 3. Config

In [ ]:
PIPELINE_OUTPUT = Path("three_crop_output")
MODEL_NAME      = "hf-hub:imageomics/bioclip-2"
CHECKPOINT_PATH = "checkpoints/best.pt"  # set to None to use pretrained weights
BATCH_SIZE      = 64
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

# Background removal — applied BEFORE CLAHE
# Set APPLY_BG_REMOVAL = False to disable
APPLY_BG_REMOVAL = True
BG_COMPOSITE_COLOR = (0, 0, 0)   # color to paste foreground onto after alpha removal
                                  # black matches the pad_to_square pad color

# CLAHE — applied to each crop (after bg removal) before embedding
# Set APPLY_CLAHE = False to disable
APPLY_CLAHE       = True
CLAHE_CLIP_LIMIT  = 3.0
CLAHE_TILE_SIZE   = (8, 8)   # smaller tiles than inference = more local contrast

# HDBSCAN
HDBSCAN_MIN_CLUSTER_SIZE = 12
HDBSCAN_MIN_SAMPLES      = 3

# UMAP
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST    = 0.1

# Visualization
SAMPLES_PER_CLUSTER  = 8
MAX_CLUSTERS_TO_SHOW = 20

print(f"Using device: {DEVICE}")
print(f"Background removal: {'ON' if APPLY_BG_REMOVAL else 'OFF'}")
print(f"CLAHE preprocessing: {'ON' if APPLY_CLAHE else 'OFF'}")

## 4. Background removal — BiRefNet session

BiRefNet gives the cleanest edges on fine animal features (fins, antennae, tentacles). On CPU-only environments rembg falls back to u2net automatically.

In [ ]:
# BiRefNet — best quality for fine animal edges; falls back to u2net on CPU
try:
    bg_session = new_session("birefnet-general")
    print("Using BiRefNet session")
except Exception as e:
    print(f"BiRefNet unavailable ({e}), falling back to u2net")
    bg_session = new_session("u2net")
    print("Fell back to u2net session")


def remove_background(pil_img: Image.Image) -> Image.Image:
    """Remove background with rembg/BiRefNet, feather the edge, return RGBA."""
    result_rgba = remove(pil_img.convert("RGB"), session=bg_session)
    w_crop, h_crop = pil_img.size

    alpha_raw = np.array(result_rgba)[:, :, 3]
    alpha_mask = cv2.resize(
        alpha_raw.astype(np.float32),
        (w_crop, h_crop),
        interpolation=cv2.INTER_LANCZOS4,
    )
    alpha_mask = np.clip(alpha_mask, 0, 255).astype(np.uint8)

    # Feather edge
    alpha_f    = cv2.GaussianBlur(alpha_mask.astype(np.float32), (3, 3), sigmaX=0.8)
    alpha_mask = np.clip(alpha_f, 0, 255).astype(np.uint8)

    rgba = pil_img.convert("RGBA")
    rgba.putalpha(Image.fromarray(alpha_mask))
    return rgba


def composite_on_color(rgba: Image.Image, bg_color=(0, 0, 0)) -> Image.Image:
    """Flatten an RGBA image onto a solid background color, returning RGB.

    BioClip's preprocess expects RGB. Black matches pad_to_square's pad color so
    the foreground sits on a uniform black canvas all the way through.
    """
    bg = Image.new("RGB", rgba.size, bg_color)
    bg.paste(rgba, mask=rgba.split()[3])
    return bg

## 5. Track ID parsing

If your filename convention changes, only edit `parse_track_id`.

In [ ]:
def parse_track_id(path: Path) -> str:
    """Extract track ID from a crop filename."""
    return path.stem.split("_")[0]   # "id3_fish_f0042_best" → "id3" 

## 6. Collect crops

In [ ]:
def collect_crop_paths(pipeline_output: Path) -> list[Path]:
    """Gather all .jpg crops from pipeline_output/<video>/"""
    paths = sorted(pipeline_output.glob("*/*.jpg"))
    print(f"Found {len(paths)} crop images across "
          f"{len(set(p.parent.name for p in paths))} video(s)")
    return paths

## 7. CLAHE preprocessing

In [ ]:
def apply_clahe(pil_img: Image.Image) -> Image.Image:
    """
    Apply CLAHE on the L channel of LAB colorspace to enhance
    local contrast without blowing out colors.
    """
    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP_LIMIT, tileGridSize=CLAHE_TILE_SIZE)
    bgr   = cv2.cvtColor(np.array(pil_img.convert("RGB")), cv2.COLOR_RGB2BGR)
    lab   = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    lab   = cv2.merge([clahe.apply(l), a, b])
    bgr   = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    return Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))

## 8. Crop loading

Order: **raw → background removal → composite on black → CLAHE → pad → CLIP preprocess**.

Running CLAHE *after* bg removal means it only stretches contrast over the animal pixels, not the reef behind it.

In [ ]:
def load_crop(path: Path) -> Image.Image:
    """Load a crop, optionally removing background then applying CLAHE."""
    img = Image.open(path).convert("RGB")
    if APPLY_BG_REMOVAL:
        rgba = remove_background(img)
        img  = composite_on_color(rgba, BG_COMPOSITE_COLOR)
    if APPLY_CLAHE:
        img = apply_clahe(img)
    return img


# Quick sanity check — visualize one crop through the pipeline
def preview_pipeline(path: Path):
    """Side-by-side preview: raw → bg removed → +CLAHE."""
    raw = Image.open(path).convert("RGB")
    stages = [("raw", raw)]
    if APPLY_BG_REMOVAL:
        rgba = remove_background(raw)
        nobg = composite_on_color(rgba, BG_COMPOSITE_COLOR)
        stages.append(("bg removed", nobg))
        current = nobg
    else:
        current = raw
    if APPLY_CLAHE:
        stages.append(("+ CLAHE", apply_clahe(current)))

    fig, axes = plt.subplots(1, len(stages), figsize=(4 * len(stages), 4))
    if len(stages) == 1:
        axes = [axes]
    for ax, (name, im) in zip(axes, stages):
        ax.imshow(im); ax.set_title(name); ax.axis("off")
    plt.tight_layout(); plt.show()

## 9. BioClip 2

In [ ]:
def load_bioclip(model_name: str, device: str, checkpoint_path: str = None):
    print(f"Loading {model_name}...")
    model, _, preprocess = open_clip.create_model_and_transforms(model_name)

    if checkpoint_path:
        print(f"Loading finetuned weights from {checkpoint_path}...")
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.visual.load_state_dict(ckpt["model_state"])
        print("Finetuned weights loaded ✓")
    else:
        print("Using pretrained weights")

    model = model.to(device).eval()
    print("BioClip 2 loaded ✓")
    return model, preprocess


def pad_to_square(img: Image.Image) -> Image.Image:
    w, h = img.size
    if w == h:
        return img
    size   = max(w, h)
    padded = Image.new("RGB", (size, size), (0, 0, 0))
    padded.paste(img, ((size - w) // 2, (size - h) // 2))
    return padded

## 10. Extract embeddings

In [ ]:
def extract_embeddings(
    paths: list[Path],
    model,
    preprocess,
    device: str,
    batch_size: int = 64,
) -> np.ndarray:
    all_emb = []
    for i in tqdm(range(0, len(paths), batch_size), desc="Extracting embeddings"):
        batch = []
        for p in paths[i:i + batch_size]:
            try:
                batch.append(preprocess(pad_to_square(load_crop(p))))
            except Exception as e:
                print(f"  Skipping {p.name}: {e}")
        if not batch:
            continue
        t = torch.stack(batch).to(device)
        with torch.no_grad(), torch.autocast(device):
            feats = model.encode_image(t)
            feats = feats / feats.norm(dim=-1, keepdim=True)
        all_emb.append(feats.cpu().float().numpy())
    return np.vstack(all_emb)

## 11. Average per track

In [ ]:
def average_per_track(
    paths: list[Path],
    embeddings: np.ndarray,
) -> tuple[list[str], np.ndarray]:
    """
    Average embeddings across all crops belonging to the same track ID.
    Returns (track_ids, averaged_embeddings).
    """
    track_embs = defaultdict(list)
    for path, emb in zip(paths, embeddings):
        tid = parse_track_id(path)
        track_embs[tid].append(emb)

    track_ids = sorted(track_embs.keys())
    averaged  = np.stack([np.mean(track_embs[tid], axis=0) for tid in track_ids])

    # re-normalise after averaging
    norms    = np.linalg.norm(averaged, axis=1, keepdims=True)
    averaged = averaged / np.where(norms == 0, 1, norms)

    print(f"Averaged to {len(track_ids)} tracks  (shape {averaged.shape})")
    return track_ids, averaged

## 12. HDBSCAN clustering

In [ ]:
def cluster_hdbscan(
    embeddings: np.ndarray,
    min_cluster_size: int = HDBSCAN_MIN_CLUSTER_SIZE,
    min_samples: int = HDBSCAN_MIN_SAMPLES,
) -> np.ndarray:
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric="euclidean",
        core_dist_n_jobs=-1,
    )
    labels = clusterer.fit_predict(embeddings)
    unique, counts = np.unique(labels, return_counts=True)
    n_clusters = len([u for u in unique if u != -1])
    n_noise    = int((labels == -1).sum())
    print(f"\nHDBSCAN → {n_clusters} clusters, {n_noise} noise points "
          f"({n_noise / len(labels):.1%})")
    for u, c in zip(unique, counts):
        tag = " (noise)" if u == -1 else ""
        print(f"  Cluster {u:>3}: {c:>4} tracks{tag}")
    if n_clusters > 1 and n_noise < len(labels):
        non_noise = labels != -1
        sil = silhouette_score(embeddings[non_noise], labels[non_noise],
                               sample_size=min(3000, non_noise.sum()))
        print(f"Silhouette (non-noise): {sil:.4f}")
    return labels

## 13. Visualization helpers

In [ ]:
def best_crop_for_track(tid: str, pipeline_output: Path) -> Path | None:
    """Return the 'best' crop image for a track ID, falling back to any crop."""
    matches = sorted(pipeline_output.glob(f"*/{tid}_*_best.jpg"))
    if matches:
        return matches[0]
    matches = sorted(pipeline_output.glob(f"*/{tid}_*.jpg"))
    return matches[0] if matches else None


def plot_clusters(
    labels: np.ndarray,
    track_ids: list[str],
    pipeline_output: Path,
    samples_per_cluster: int = SAMPLES_PER_CLUSTER,
    max_clusters: int = MAX_CLUSTERS_TO_SHOW,
):
    unique = np.unique(labels)
    show   = [l for l in sorted(unique) if l != -1][:max_clusters]

    fig, axes = plt.subplots(
        len(show), samples_per_cluster,
        figsize=(samples_per_cluster * 1.8, len(show) * 2.2)
    )
    if len(show) == 1:
        axes = [axes]

    for row, lbl in enumerate(show):
        idxs   = np.where(labels == lbl)[0]
        chosen = np.random.choice(idxs, size=min(samples_per_cluster, len(idxs)), replace=False)
        for col in range(samples_per_cluster):
            ax = axes[row][col]
            ax.axis("off")
            if col < len(chosen):
                tid  = track_ids[chosen[col]]
                path = best_crop_for_track(tid, pipeline_output)
                if path:
                    try:
                        ax.imshow(Image.open(path).convert("RGB"))
                    except Exception:
                        pass
            if col == 0:
                ax.set_title(f"Cluster {lbl}  (n={len(idxs)})",
                             loc="left", fontsize=9, fontweight="bold", pad=4)

    plt.suptitle("HDBSCAN clusters", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()


def plot_all_in_cluster(
    lbl: int,
    labels: np.ndarray,
    track_ids: list[str],
    pipeline_output: Path,
    cols: int = 8,
):
    idxs = np.where(labels == lbl)[0]
    tag  = "Noise" if lbl == -1 else f"Cluster {lbl}"
    print(f"{tag}: {len(idxs)} tracks")

    rows = (len(idxs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.8, rows * 2.2))
    axes = np.array(axes).reshape(-1, cols)

    for i, idx in enumerate(idxs):
        ax   = axes[i // cols][i % cols]
        ax.axis("off")
        tid  = track_ids[idx]
        path = best_crop_for_track(tid, pipeline_output)
        if path:
            try:
                ax.imshow(Image.open(path).convert("RGB"))
                ax.set_title(tid, fontsize=5, pad=2)
            except Exception:
                pass

    for j in range(len(idxs), rows * cols):
        axes[j // cols][j % cols].axis("off")

    plt.suptitle(f"{tag} — all {len(idxs)} tracks", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()


def plot_umap(embeddings: np.ndarray, labels: np.ndarray):
    print("Running UMAP...")
    reducer = umap.UMAP(
        n_components=2, metric="cosine",
        n_neighbors=UMAP_N_NEIGHBORS, min_dist=UMAP_MIN_DIST,
        random_state=42, verbose=False,
    )
    coords  = reducer.fit_transform(embeddings)
    unique  = np.unique(labels)
    palette = sns.color_palette("tab20", n_colors=len(unique))
    cmap    = {lbl: palette[i] for i, lbl in enumerate(sorted(unique))}
    colors  = [cmap[l] for l in labels]

    plt.figure(figsize=(12, 8))
    plt.scatter(coords[:, 0], coords[:, 1], c=colors, s=8, alpha=0.7, linewidths=0)
    patches = [mpatches.Patch(color=cmap[l],
                               label=f"Cluster {l}" + (" (noise)" if l == -1 else ""))
               for l in sorted(unique)]
    plt.legend(handles=patches, bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
    plt.title(f"BioClip 2 — UMAP  (HDBSCAN, "
              f"{len([u for u in unique if u != -1])} clusters)", fontsize=13)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

## 14. Run the pipeline

Each step lives in its own cell so you can re-run pieces without reloading the model or re-extracting embeddings.

In [ ]:
# 14a. Collect paths
crop_paths = collect_crop_paths(PIPELINE_OUTPUT)
assert crop_paths, f"No crops found under {PIPELINE_OUTPUT}" 

In [ ]:
# 14b. (Optional) preview the preprocessing pipeline on a single crop
preview_pipeline(crop_paths[0])

In [ ]:
# 14c. Load BioClip 2
model, preprocess = load_bioclip(MODEL_NAME, DEVICE, CHECKPOINT_PATH)

In [ ]:
# 14d. Extract per-crop embeddings
per_crop_embs = extract_embeddings(crop_paths, model, preprocess, DEVICE, BATCH_SIZE)
per_crop_embs.shape

In [ ]:
# 14e. Average per track
track_ids, track_embs = average_per_track(crop_paths, per_crop_embs)

In [ ]:
# 14f. Cluster
labels = cluster_hdbscan(track_embs)

In [ ]:
# 14g. Cluster samples
plot_clusters(labels, track_ids, PIPELINE_OUTPUT)

In [ ]:
# 14h. Inspect noise cluster
if -1 in labels:
    plot_all_in_cluster(-1, labels, track_ids, PIPELINE_OUTPUT)

In [ ]:
# 14i. Inspect any large cluster (>100 tracks)
unique, counts = np.unique(labels, return_counts=True)
for lbl, count in zip(unique, counts):
    if lbl != -1 and count > 100:
        plot_all_in_cluster(lbl, labels, track_ids, PIPELINE_OUTPUT)

In [ ]:
# 14j. UMAP
plot_umap(track_embs, labels)